In [1]:
!pip install unsloth
!pip install transformers datasets accelerate peft trl bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.7/842.7 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024 # Reduced from 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.2-bnb-4bit", # Changed to a smaller model
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.4: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.4 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [4]:
from datasets import Dataset

data = [
    {"text": "Q: What are symptoms of diabetes?\nA: Frequent urination, increased thirst, and weight loss."},
    {"text": "Q: What is hypertension?\nA: A condition where blood pressure is consistently too high."},
    {"text": "Q: What causes asthma?\nA: Airway inflammation triggered by allergens or irritants."},
    {"text": "Q: What is fever?\nA: Temporary increase in body temperature due to infection."},
    {"text": "Q: What is anemia?\nA: A condition where there are not enough healthy red blood cells."},
]

dataset = Dataset.from_list(data)

In [5]:
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=max_seq_length)

tokenized_dataset = dataset.map(tokenize_function)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = tokenized_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none"
    ),
)

In [7]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5 | Num Epochs = 30 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 13,631,488 of 7,255,363,584 (0.19% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,11.169141
20,5.545526
30,4.926511
40,4.880234
50,4.869183
60,4.864496


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-60.


TrainOutput(global_step=60, training_loss=6.042515055338542, metrics={'train_runtime': 331.5957, 'train_samples_per_second': 0.724, 'train_steps_per_second': 0.181, 'total_flos': 6565747123814400.0, 'train_loss': 6.042515055338542, 'epoch': 30.0})

In [8]:
model.save_pretrained("medical_lora_adapter")
tokenizer.save_pretrained("medical_lora_adapter")

Unsloth: Restored added_tokens_decoder metadata in medical_lora_adapter/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in medical_lora_adapter.


('medical_lora_adapter/tokenizer_config.json',
 'medical_lora_adapter/chat_template.jinja',
 'medical_lora_adapter/tokenizer.json')

In [9]:
model.config.max_length = None

In [14]:
# Cell 6: Strict Clean Inference for Asthma Symptoms (Short & Clean)
import os
import warnings
import logging

# Suppress all Hugging Face and logging warnings completely
os.environ["HF_LOGGING_LEVEL"] = "ERROR"
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

import torch
from unsloth import FastLanguageModel

# Set model to inference mode cleanly
FastLanguageModel.for_inference(model)

# Updated Short Query for Asthma Symptoms
clinical_query = "What are the common symptoms of asthma?"

inputs = tokenizer(
    [train_prompt_style.format(clinical_query, "", "", "")],
    return_tensors = "pt"
).to("cuda")

# Generation with constraints for a short, crisp answer
outputs = model.generate(
    **inputs,
    max_new_tokens = 60,     # Kept small for short response
    use_cache = True,
    pad_token_id = tokenizer.eos_token_id
)

decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Print ONLY the clean requested output block
print("=== CLEAN FINE-TUNED MEDICAL CHATBOT OUTPUT ===")
print(decoded_output)

=== CLEAN FINE-TUNED MEDICAL CHATBOT OUTPUT ===
Q: What are the common symptoms of asthma?
A: Wheezing, shortness of breath, chest tightness, coughing.
